# 🏁 Compare — every strategy against its buy-and-hold baseline

The cross-strategy scoreboard. Every run below follows the repo's
**comparability rules** (see the root README): ONE shared **$10k** wallet per
strategy, fractional shares, the same global cost model, and returns reported
as **geometric CAGR** (profits stay reinvested; the reported rate is the
constant yearly rate that compounds to the same endpoint).

For each strategy this notebook loads the panel its `config.yaml` asks for,
runs the strategy at its **config-default parameters**, and runs
`strategies/baseline` — equal-weight buy-and-hold, the engine hurdle made
visible — on the *same panel*. `excess_cagr` and `info_ratio` measure the
distance between the two; zero means "you could have just bought everything".

> Strategies are tuned on different universes and windows, so compare each
> strategy **to its own baseline row**, not raw numbers across universes.
> `years` rides along to show how much sample sits behind each number.

## 1 · Setup

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, strategies
from trading_simple import plots
from trading_simple.data import load_panel, symbols_with_bars

ref_fees, ref_slip = bt.leg_costs(settings.costs)   # $20-name reference, printout only —
INIT_CASH = settings.backtest.init_cash             # real runs price at each panel's median
print(f"init_cash: ${INIT_CASH:,.0f} (one shared wallet per strategy)  "
      f"reference fees/leg: {ref_fees:.4%}  slippage/leg: {ref_slip:.4%}")
print("strategies:", ", ".join(strategies.available()))

## 2 · Run every strategy and its baseline

One panel per distinct `data:` block (cached, so strategies sharing a
universe/window also share the exact panel — and the baseline runs once per
panel, not once per strategy).

In [ ]:
_panels: dict[tuple, object] = {}


def panel_for(strat):
    """Load (and cache) the panel a strategy's `data:` block asks for."""
    d = strat.data
    key = (d["universe"], d["bar"], d.get("resample"), d["start"], d.get("end"),
           d.get("session", "regular"))
    if key not in _panels:
        syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                                 d["bar"], d["start"], d.get("end"),
                                 limit=d.get("max_symbols"))
        panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                           session=d.get("session", "regular"), exchanges=settings.exchanges,
                           resample=d.get("resample"))
        if d.get("features"):  # alt-data strategies (informed_flow)
            from trading_simple import features
            features.attach_features(panel, engine, d["features"], settings=settings)
        _panels[key] = panel
    return _panels[key]


def stats_and_curve(strat, panel, start=None):
    """One backtest -> the evaluate() stats dict plus the $-equity curve."""
    # Warm from `start` (a study's optuna.score_from): signals on the whole panel, book and
    # metrics on the scored bars only — the window the strategy's own numbers are measured on.
    pf, scored, fees, slippage = bt._run_scored(strat, panel, None, settings, start)
    stats = bt.portfolio_stats(pf)
    stats.update(bt.active_stats(pf, scored, fees=fees, slippage=slippage,
                                 fill=strat.fill, total_return=stats["total_return"]))
    return stats, bt.equity_curve(pf, init_cash=INIT_CASH)["value"]


baseline = strategies.load("baseline")
rows, curves = [], {}   # curves[(universe, bar, window)] = {label: equity series}

for name in strategies.available():
    if name == "baseline":     # runs once per panel below, not as its own row here
        continue
    strat = strategies.load(name)
    panel = panel_for(strat)
    d = strat.data
    start = strat.optuna.get("score_from")    # e.g. a universe that is point-in-time from there
    window = f"{start or d['start']} → {d.get('end') or 'now'}"
    key = (d["universe"], panel.bar, window)   # window matters: same universe, other
                                              # dates = a different baseline

    stats, curve = stats_and_curve(strat, panel, start)
    rows.append({"strategy": name, "universe": d["universe"], "bar": panel.bar,
                 "window": window, **stats})
    curves.setdefault(key, {})[name] = curve

    if "baseline" not in curves[key]:
        bstats, bcurve = stats_and_curve(baseline, panel, start)
        rows.append({"strategy": f"baseline @ {d['universe']}/{panel.bar}",
                     "universe": d["universe"], "bar": panel.bar,
                     "window": window, **bstats})
        curves[key]["baseline"] = bcurve
    print(f"{name:20s} {d['universe']:>16s}/{panel.bar}  "
          f"excess_cagr {stats['excess_cagr']:+7.1%}  info_ratio {stats['info_ratio']:+5.2f}")

## 3 · Scoreboard

All numbers per **train window** at **config-default parameters** — the tuned
picks live in each strategy's own notebook, which is also where the holdout
verdict is. `cagr`/`excess_cagr` are geometric and annualized; a baseline row's
`excess_cagr` is ~0 by construction (it *is* the benchmark).

In [ ]:
import pandas as pd

table = pd.DataFrame(rows).sort_values("excess_cagr", ascending=False)
cols = ["strategy", "universe", "bar", "window", "years", "cagr", "excess_cagr",
        "alpha_cagr", "beta", "info_ratio", "sharpe", "max_drawdown", "n_trades",
        "trades_per_year"]
# strategy stays a column (the same baseline label can appear once per window)
table[cols].style.format({
    "years": "{:.2f}", "cagr": "{:+.1%}", "excess_cagr": "{:+.1%}",
    "alpha_cagr": "{:+.1%}", "beta": "{:.2f}",
    "info_ratio": "{:+.2f}", "sharpe": "{:.2f}", "max_drawdown": "{:.1%}",
    "n_trades": "{:,.0f}", "trades_per_year": "{:,.0f}",
}).background_gradient(subset=["excess_cagr"], cmap="RdBu", vmin=-1.0, vmax=1.0)\
 .hide(axis="index")

## 4 · Equity overlays — strategy vs baseline, per panel

Cumulative return of the same $10k wallet. The dashed gray line is the
baseline (equal-weight buy-and-hold of that panel); finishing above it is the
whole game. Long 1m series are envelope-thinned by `plots._scatter`, which
keeps every spike and drawdown low visible.

In [ ]:
import plotly.graph_objects as go

for (universe, bar, window), group in curves.items():
    fig = go.Figure()
    fig.add_trace(plots._scatter(
        group["baseline"] / INIT_CASH - 1.0, name="baseline (buy & hold)",
        line={"color": "#898781", "dash": "dash", "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>baseline</extra>"))
    strats = [n for n in sorted(group) if n != "baseline"]
    for i, name in enumerate(strats):
        fig.add_trace(plots._scatter(
            group[name] / INIT_CASH - 1.0, name=name,
            line={"color": plots.SERIES[i % len(plots.SERIES)], "width": 1.5},
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + name + "</extra>"))
    plots._base(fig, title=f"{universe} · {bar} · {window} — cumulative return "
                           "vs buy-and-hold", height=420)
    fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
    fig.show()

## Read the result honestly

- These are **train-window** numbers at **config defaults** — optimistic by
  construction for tuned strategies, and no substitute for each strategy
  notebook's out-of-sample (§11) and cost-stress (§12) sections.
- A strategy can only be judged against **its own baseline row**: the
  universes differ, and a hot universe sets a brutally high buy-and-hold hurdle
  that has nothing to do with strategy skill. **This is not a small effect and
  it is measured.** `HighOctaneTop20` — the universe five 1m strategies here
  still use — was pinned in 2026-07 by trailing-90-day dollar volume, i.e.
  *selected using data from inside and after* the 2025-07 → 2026-04 window it
  is tested on, and its equal-weight buy-and-hold returned **+460%/yr** over
  that window. A morning-only strategy scoring `excess_cagr ≈ −4.6` against it
  has been told nothing about itself. `gap_and_go` has migrated to the
  point-in-time `Morning200` (hurdle +47%/yr); the rest have not, and their
  rows should be read with that in mind. See
  `notebooks/01_universe_screener.ipynb` §8 and each strategy's `IDEAS.md`.
- Short windows annualize to loud numbers — check `years` before believing
  any `cagr`. Annualizing makes a short backtest *comparable*, never
  *trustworthy*.
- Two 1m panels of 200+ symbols do not fit alongside everything else on a
  16 GB machine. If §2 dies, restart the kernel and run a subset of
  `strategies.available()`.